In [ ]:
import os
import pandas as pd
from torchvision.io import decode_image
from torch.utils.data import Dataset

In [ ]:
df = pd.read_csv('../data/train_ship_segmentations_v2.csv')
df['EncodedPixels'] = df['EncodedPixels'].fillna('')
df

In [ ]:
for i in range(10):
    print(df['ImageId'][i], df['EncodedPixels'][i])

In [ ]:
import sys
sys.path.append("..")

from src.rle import rle_to_mask

class AirbusShipDetectionDataset(Dataset):
    def __init__(self, masks_file, img_dir):
        self.masks_file = masks_file
        self.img_dir = img_dir
        
        self.df = pd.read_csv(masks_file)
        self.df['EncodedPixels'] = self.df['EncodedPixels'].fillna('')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_file = self.df['ImageId'][idx]
        img_path = os.path.join(self.img_dir, img_file)
        image = decode_image(img_path)

        rle = self.df['EncodedPixels'][idx]
        _, h, w = image.shape
        mask = rle_to_mask(rle, h, w)

        return image, mask


In [ ]:
from torch.utils.data import DataLoader

dataset = AirbusShipDetectionDataset(masks_file='../data/train_ship_segmentations_v2.csv', img_dir='../data/train_v2')
loader = DataLoader(dataset, batch_size=2)

img, mask = next(iter(loader))
img, mask

In [ ]:
import sys
sys.path.append("..")

from src.data import loader

l = loader('../data')
l